# 01 – assembly_summary.csv (historian)

**Goal:** understand this one table well enough to explain it. Look and count only, no cleaning (cleaning is silver, Phase 2).

**Done when I can answer without looking:**
1. What does one row represent?
2. What does each column mean?
3. What problems does the data have, and how many rows does each affect?

---

## 1. One row
- One filter assembly's integrity tests: Pre-IT and PUPSIT.
- No batch ID. Must match to MES batches by time window and line.
- 2834 rows in files → **2776 real assemblies** (after removing 27 junk rows and 31 duplicate copies).

## 2. Columns (two formats)
Schema change on **1 Oct 2025** (historian upgrade).

| Meaning | Old (before 1 Oct 2025) | New (from 1 Oct 2025) |
|---|---|---|
| Test cycle time | `FIT_Cycle_Window` (one string `"start - end"`) | `Cycle_Start_UTC` + `Cycle_End_UTC` |
| Cart | `Cart` | `Equipment` |
| Pre-IT flow | `PreIT_Flow` | `PreIT_DiffFlow_mL_min` |
| PUPSIT flow | `PUPSIT_Flow` | `PUPSIT_DiffFlow_mL_min` |
| Pre-IT wetting time | `Wetting_Time` (text `"85 s"`) | `PreIT_Wetting_Time_s` (number) |
| PUPSIT wetting time | `PUPSIT_Wetting_Time` (text) | `PUPSIT_Wetting_Time_s` (number) |
| Pre-IT attempts | `Attempts` | `PreIT_Attempts` |
| PUPSIT attempts | `PUPSIT_Attempts` | `PUPSIT_Attempts` (unchanged) |

- Old names `Wetting_Time` and `Attempts` mean **Pre-IT only**; the name does not say so.
- README misses two renames: `Attempts` → `PreIT_Attempts`, `PUPSIT_Flow` → `PUPSIT_DiffFlow_mL_min`.

## 3. Problems found
| Problem | Rows | Notes |
|---|---|---|
| Two header formats | 1403 old, 1431 new (2834 total) | `pd.concat` gives NaN in the other format's columns |
| Missing written as `-` (old) | 45 in `PreIT_Flow` | `isna()` does not count `-`; use `na_values=["-"]` |
| Missing left blank (new) | 10 in wetting columns | these are the 10 new-format junk rows |
| Junk rows (attempts 0) | 27 (17 old, 10 new) | all exactly 12 min long, results all `-` / blank; real rows 4.9–6.7 h → quarantine |
| Pre-IT flow `-` but attempts ≥ 1 | 28 | test ran but value not recorded |
| Flow `0` | 0 | README says `0` = aborted, but not seen in this file |
| Cart name written 4 ways | see table below | same 6 carts (A1, A2, B1, B2, C1, C2); all 4 forms in both `Cart` and `Equipment` |
| Duplicates | 31 copies (~1.1%) | resent in next day's drop; only 13 exact copies, 18 have a different cart form → dedupe on start + normalised cart |
| Late rows | 48 of 2776 (~1.7%) | arrive 1–3 days after the test ended → silver must accept past dates |
| Line A outage | 3 drops (28–30 Jul 2025) | no Line A rows, backfilled in 1 Aug drop |

**Cart name forms** (example for cart A1):

| Form | Example | `Cart` (old) | `Equipment` (new) |
|---|---|---|---|
| Plain | `A1` | 791 | 780 |
| Underscore | `FITC_A1` | 364 | 376 |
| Dash | `FITC-A1` | 124 | 130 |
| With word "Cart" | `Cart A1` | 124 | 145 |
| **Total** | | 1403 | 1431 |

- `value_counts()` treats each form as a different cart, so counts per cart are split 4 ways until the names are mapped to one form (silver, via `reference/equipment_master.csv`).

## Checklist
- [x] Cart names written 4 ways
- [x] Attempts-0 rows are junk (check A)
- [x] Duplicate rows (check B)
- [x] Late rows (check C)
- [x] Line A outage and backfill (check D)
- [ ] Say back the 3 "done when" answers without looking
- [ ] Open a PR for this notebook

In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",None)

import os

In [2]:
# file_path = os.path.join("..", "data", "raw", "drops", "2024-10-14", "historian", "assembly_summary.csv")

# file_data = pd.read_csv(file_path,dtype='str',keep_default_na=True)
# file_data.head()



In [3]:
# i need to concatenate all summary_assembly files in a datafrae
# i need to iterate in drops directory
# then in each folder look for historian folder
# inside historian folder look for summary_assembly.csv file
# load it and append it to a dataframe
master_df = pd.DataFrame()
for root, dirs, files in os.walk(os.path.join("..", "data", "raw", "drops")):
    if "historian" in dirs:
        historian_path = os.path.join(root,"historian")
        # print(historian_path)
        if "assembly_summary.csv" in os.listdir(historian_path):
            file_path = os.path.join(historian_path, "assembly_summary.csv")
            file_data = pd.read_csv(file_path, dtype='str', keep_default_na=True)
            # Append to a master dataframe
            master_df = pd.concat([master_df, file_data], ignore_index=True)




In [4]:
master_df.head()

,FIT_Cycle_Window,Cart,PreIT_Flow,PUPSIT_Flow,Wetting_Time,PUPSIT_Wetting_Time,Attempts,PUPSIT_Attempts,Cycle_Start_UTC,Cycle_End_UTC,Equipment,PreIT_DiffFlow_mL_min,PUPSIT_DiffFlow_mL_min,PreIT_Wetting_Time_s,PUPSIT_Wetting_Time_s,PreIT_Attempts
0,2024-10-01 06:54:41 - 2024-10-01 12:59:07,B2,0.84,3.22,108 s,91 s,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-10-01 08:14:34 - 2024-10-01 14:19:10,C1,0.757,2.907,106 s,105 s,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-10-01 08:18:56 - 2024-10-01 14:25:26,C2,0.818,2.816,85 s,90 s,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-10-01 07:28:05 - 2024-10-01 13:00:07,A1,0.663,2.808,108 s,104 s,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-10-01 07:30:22 - 2024-10-01 12:42:08,FITC-A2,0.823,2.805,98 s,85 s,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
master_df.isna().sum()

FIT_Cycle_Window          1431
Cart                      1431
PreIT_Flow                1431
PUPSIT_Flow               1431
Wetting_Time              1431
PUPSIT_Wetting_Time       1431
Attempts                  1431
PUPSIT_Attempts              0
Cycle_Start_UTC           1403
Cycle_End_UTC             1403
Equipment                 1403
PreIT_DiffFlow_mL_min     1403
PUPSIT_DiffFlow_mL_min    1403
PreIT_Wetting_Time_s      1413
PUPSIT_Wetting_Time_s     1413
PreIT_Attempts            1403
dtype: int64

In [6]:
master_df['PreIT_Flow'].value_counts()

PreIT_Flow
-        45
0.757    12
0.695    12
0.748    12
0.7      12
0.739    12
0.797    11
0.795    11
0.76     11
0.66     10
0.683    10
0.798    10
0.709     9
0.791     9
0.806     9
0.72      9
0.751     9
0.681     9
0.755     9
0.778     8
0.771     8
0.669     8
0.742     8
0.792     8
0.721     8
0.726     8
0.77      8
0.793     8
0.75      8
0.68      8
0.756     8
0.708     8
0.637     8
0.718     8
0.698     8
0.777     8
0.731     8
0.818     7
0.719     7
0.713     7
0.781     7
0.682     7
0.706     7
0.728     7
0.727     7
0.737     7
0.69      7
0.699     7
0.765     7
0.768     7
0.676     7
0.686     7
0.715     7
0.758     7
0.673     7
0.783     7
0.762     7
0.759     7
0.663     6
0.712     6
0.775     6
0.749     6
0.79      6
0.645     6
0.666     6
0.691     6
0.661     6
0.815     6
0.817     6
0.724     6
0.734     6
0.82      6
0.732     6
0.677     6
0.773     6
0.733     6
0.836     6
0.746     6
0.685     6
0.723     6
0.678     6
0.803     6
0.687

In [7]:
a = master_df['PreIT_Flow'] == 0
a.sum()

np.int64(0)

In [8]:
master_df[['Equipment','Cart']].value_counts(dropna=False)

Equipment  Cart   
NaN        C2         141
           A1         140
B2         NaN        138
C2         NaN        136
C1         NaN        135
NaN        A2         131
           B2         129
A2         NaN        128
NaN        C1         127
A1         NaN        125
NaN        B1         123
B1         NaN        118
NaN        FITC_C1     74
FITC_B1    NaN         74
NaN        FITC_B1     66
FITC_B2    NaN         64
FITC_C1    NaN         62
NaN        FITC_A2     61
FITC_A1    NaN         61
FITC_A2    NaN         58
NaN        FITC_B2     57
FITC_C2    NaN         57
NaN        FITC_A1     55
           FITC_C2     51
           FITC-B1     29
Cart C2    NaN         29
NaN        Cart C2     27
FITC-A2    NaN         27
Cart A2    NaN         27
FITC-A1    NaN         25
Cart B2    NaN         24
NaN        Cart B2     22
           FITC-B2     22
           Cart B1     22
Cart B1    NaN         22
Cart A1    NaN         22
FITC-C2    NaN         22
NaN        FITC-A2 

In [13]:
master_df.columns

Index(['FIT_Cycle_Window', 'Cart', 'PreIT_Flow', 'PUPSIT_Flow', 'Wetting_Time',
       'PUPSIT_Wetting_Time', 'Attempts', 'PUPSIT_Attempts', 'Cycle_Start_UTC',
       'Cycle_End_UTC', 'Equipment', 'PreIT_DiffFlow_mL_min',
       'PUPSIT_DiffFlow_mL_min', 'PreIT_Wetting_Time_s',
       'PUPSIT_Wetting_Time_s', 'PreIT_Attempts'],
      dtype='str')

In [20]:
# 1. check rows where attempts = 0
# 2. check time of rows where attempts is 0

cols = ['Cycle_Start_UTC','Cycle_End_UTC']
master_df[cols] = master_df[cols].apply(pd.to_datetime)

zero_preIT_attempts = master_df[master_df['PreIT_Attempts']=='0']

zero_preIT_attempts['Cycle_End_UTC'] - zero_preIT_attempts['Cycle_Start_UTC']

1407   0 days 00:12:00
1464   0 days 00:12:00
1596   0 days 00:12:00
1600   0 days 00:12:00
1849   0 days 00:12:00
1902   0 days 00:12:00
1957   0 days 00:12:00
1997   0 days 00:12:00
2388   0 days 00:12:00
2428   0 days 00:12:00
dtype: timedelta64[us]

In [14]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2834 entries, 0 to 2833
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   FIT_Cycle_Window        1403 non-null   str                
 1   Cart                    1403 non-null   str                
 2   PreIT_Flow              1403 non-null   str                
 3   PUPSIT_Flow             1403 non-null   str                
 4   Wetting_Time            1403 non-null   str                
 5   PUPSIT_Wetting_Time     1403 non-null   str                
 6   Attempts                1403 non-null   str                
 7   PUPSIT_Attempts         2834 non-null   str                
 8   Cycle_Start_UTC         1431 non-null   datetime64[us, UTC]
 9   Cycle_End_UTC           1431 non-null   datetime64[us, UTC]
 10  Equipment               1431 non-null   str                
 11  PreIT_DiffFlow_mL_min   1431 non-null   str           

## Remaining checks

Reload all files with one extra column, `drop_date` (the folder the row arrived in). We need it for late rows, duplicates and the outage. Then add helper columns that work for both formats:
- `start`, `end`: old format split from `FIT_Cycle_Window`, new format from `Cycle_Start_UTC` / `Cycle_End_UTC`
- `duration`: `end - start`
- `pre_attempts`: `Attempts` (old) or `PreIT_Attempts` (new)
- `cart`: last 2 characters of `Cart` / `Equipment` (`FITC_A1`, `FITC-A1`, `Cart A1`, `A1` all become `A1`)
- `line`: first letter of `cart`

In [ ]:
import glob

drops_path = os.path.join("..", "data", "raw", "drops")
frames = []
for file_path in sorted(glob.glob(os.path.join(drops_path, "*", "historian", "assembly_summary.csv"))):
    drop_date = file_path.split(os.sep)[-3]          # ..\data\raw\drops\<drop_date>\historian\file.csv
    frames.append(pd.read_csv(file_path, dtype="str").assign(drop_date=drop_date))
df = pd.concat(frames, ignore_index=True)            # concat once, after the loop
df["drop_date"] = pd.to_datetime(df["drop_date"])

# start / end for both formats
window = df["FIT_Cycle_Window"].str.split(" - ", expand=True)   # old: "start - end" -> 2 columns
df["start"] = pd.to_datetime(window[0]).dt.tz_localize("UTC").fillna(pd.to_datetime(df["Cycle_Start_UTC"]))
df["end"] = pd.to_datetime(window[1]).dt.tz_localize("UTC").fillna(pd.to_datetime(df["Cycle_End_UTC"]))
df["duration"] = df["end"] - df["start"]

df["pre_attempts"] = df["Attempts"].fillna(df["PreIT_Attempts"])
df["cart"] = df["Cart"].fillna(df["Equipment"]).str[-2:]
df["line"] = df["cart"].str[0]

print(len(df), "rows")
df[["drop_date", "start", "end", "duration", "pre_attempts", "cart", "line"]].head()

In [ ]:
# Check A: are attempts-0 rows junk? Compare duration with normal rows.
df["is_empty_row"] = df["pre_attempts"] == "0"
df.groupby("is_empty_row")["duration"].describe()[["count", "min", "mean", "max"]]

In [ ]:
# Check B: duplicate rows. Same assembly = same start time + same cart.
# keep="first" marks only the later copies as duplicates.
df["is_duplicate"] = df.duplicated(subset=["start", "cart"], keep="first")
print("duplicate copies:", df["is_duplicate"].sum())

# Are the copies identical apart from drop_date? And how many days later did they arrive?
dups = df[df.duplicated(subset=["start", "cart"], keep=False)].sort_values(["start", "cart", "drop_date"])
data_cols = [c for c in df.columns if c not in ("drop_date", "is_duplicate")]
print("copies differ in data columns:", (dups.groupby(["start", "cart"])[data_cols].nunique(dropna=False) > 1).any(axis=None))
dups.groupby(["start", "cart"])["drop_date"].agg(lambda s: (s.max() - s.min()).days).value_counts()

In [ ]:
# Check C: late rows. Lag = drop date minus the date the test ended.
# Use real rows only (no empty rows, no duplicate copies), so each assembly counts once.
real = df[~df["is_empty_row"] & ~df["is_duplicate"]].copy()
real["lag_days"] = (real["drop_date"] - real["end"].dt.tz_localize(None).dt.normalize()).dt.days
print(real["lag_days"].value_counts().sort_index())
print("late (lag >= 1):", (real["lag_days"] >= 1).sum(), "of", len(real))
real[real["lag_days"] >= 1].groupby("line").size()

In [ ]:
# Check D: Line A outage. Rows per drop for line A around day 300 (~28 Jul 2025), all lines for comparison.
around = df[(df["drop_date"] >= "2025-07-26") & (df["drop_date"] <= "2025-08-02")]
print(around.groupby(["drop_date", "line"]).size().unstack(fill_value=0))

# What arrived in the 1 Aug drop, and when did those tests end?
around[around["drop_date"] == "2025-08-01"][["cart", "end", "drop_date"]]

## Conclusions of checks A–D

**A. Attempts-0 rows are junk.** 27 rows (17 old, 10 new). Every one lasts exactly 12 minutes. Real rows last 4 h 53 min to 6 h 43 min (mean 5 h 44 min). Not real assemblies → quarantine in silver, do not report as "test not done".

**B. Duplicates: 31 rows** (~1.1%). Same assembly (same start + cart) delivered again in the **next day's drop**, always exactly 1 day later.
- Only 13 of the 31 are exact copies. In the other 18 the **cart name is written in a different form** in the copy (e.g. `FITC_A1` then `A1`).
- So `df.drop_duplicates()` on full rows misses most of them. Dedupe on a key (start time + normalised cart), after cart names are mapped.

**C. Late rows: 48 of 2776 real rows** (~1.7%) arrive 1–3 days after the test ended (15 one day, 9 two days, 24 three days). All lines: A 23, B 15, C 10. Line A is higher because it includes the outage backfill below.
- Consequence: a drop does not contain everything that happened that day. Silver must be incremental and accept rows for past dates.

**D. Line A outage: 28–30 Jul 2025** (day 300). Line A has 0 rows in those 3 drops while B and C continue. The 1 Aug drop has 5 Line A rows, 3 of them from tests that ended 29–30 Jul (backfill).
- Line A also has many normal days with 0 rows (about one assembly per cart every 2 days, plus shutdowns at Christmas), so "0 rows" alone does not prove an outage. Compare with the other lines.

**Real assemblies:** 2834 rows − 27 junk − 31 duplicate copies = **2776**.